In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from pathlib import Path
from google.colab import files
import io, os, warnings
warnings.filterwarnings("ignore")

print("Upload BOTH files when the dialog opens:")
print("  • 3D Full Body Humain Gait Walking Dataset (Noisy Values).csv")
print("  • 3D Full Body Humain Gait Walking Dataset (True Values).csv")
uploaded = files.upload()

noisy_key = [k for k in uploaded if "Noisy" in k or "noisy" in k][0]
true_key  = [k for k in uploaded if "True"  in k or "true"  in k][0]
print(f"\n✅ Noisy : {noisy_key}")
print(f"✅ True  : {true_key}")



Colab link --> https://colab.research.google.com/drive/1ka6Yg2UxB-5xT6nz_byi80V3PQZxjbTt

In [ ]:
JOINT_NAMES = [
    "pelvis",       # 0
    "L5",           # 1
    "L3",           # 2
    "T12",          # 3
    "T8",           # 4
    "neck",         # 5
    "head",         # 6
    "shoulderRight",# 7
    "upperArmRight",# 8
    "forearmRight", # 9
    "handRight",    # 10
    "shoulderLeft", # 11
    "upperArmLeft", # 12
    "forearmLeft",  # 13
    "handLeft",     # 14
    "upperLegRight",# 15
    "lowerLegRight",# 16
    "footRight",    # 17
    "toeRight",     # 18
    "upperLegLeft", # 19
    "lowerLegLeft", # 20
    "footLeft",     # 21
    "toeLeft",      # 22
]
NUM_JOINTS = len(JOINT_NAMES)   # 23

# ── USER-EDITABLE SETTINGS ─────────────────────────────────────────────
CHOSEN_JOINT = 0    # 0 = pelvis  |  change to any index 0-22 or name string
FPS          = 30   # frames per second of your capture system
# ───────────────────────────────────────────────────────────────────────

def resolve_joint(arg):
    if isinstance(arg, int): return arg
    try:    return int(arg)
    except ValueError: pass
    for i, n in enumerate(JOINT_NAMES):
        if n.lower() == str(arg).lower(): return i
    raise ValueError(f"Unknown joint '{arg}'")

JOINT_IDX  = resolve_joint(CHOSEN_JOINT)
JOINT_NAME = JOINT_NAMES[JOINT_IDX]
print(f"✅ Analysing Joint {JOINT_IDX}: '{JOINT_NAME}'  |  FPS={FPS}")


In [ ]:
def load_xyz_csv(raw_bytes, num_joints=NUM_JOINTS):
    """
    Load a gait CSV (with or without text header).
    Returns ndarray shape (N_frames, num_joints, 3).
    """
    df = pd.read_csv(io.BytesIO(raw_bytes), header=None, dtype=str)

    # Detect & drop header row
    try:
        float(df.iloc[0, 0])
    except (ValueError, TypeError):
        df = df.iloc[1:].reset_index(drop=True)

    cols_needed = num_joints * 3
    data = df.iloc[:, :cols_needed].astype(float).values
    assert data.shape[1] == cols_needed, \
        f"Expected {cols_needed} cols, got {data.shape[1]}"
    N = data.shape[0]
    return data.reshape(N, num_joints, 3)

noisy_arr = load_xyz_csv(uploaded[noisy_key])   # (N, 23, 3)
true_arr  = load_xyz_csv(uploaded[true_key])    # (N, 23, 3)

N_noisy, N_true = noisy_arr.shape[0], true_arr.shape[0]
print(f"✅ Noisy frames : {N_noisy}")
print(f"✅ True  frames : {N_true}")


In [ ]:
%%writefile io.h
#ifndef IO_H
#define IO_H

#include <stdio.h>

#define MAX_LINE_LENGTH 8192
#define MAX_FRAMES 10000

// Data structure for loaded motion data
typedef struct {
    double** frames;     // [num_frames][num_joints * 3]
    int num_frames;
    int num_joints;
} MotionData;

// I/O Functions
MotionData* io_load_csv(const char* filename, int num_joints);
void io_free_motion_data(MotionData* data);

FILE* io_open_output_csv(const char* filename, int num_joints, int state_dim);
void io_write_state_row(FILE* fp, int frame_idx, const double* state, int total_dim);
void io_close_csv(FILE* fp);

#endif


In [ ]:
%%writefile io.c
#include "io.h"
#include "utils.h"
#include <stdlib.h>
#include <string.h>

MotionData* io_load_csv(const char* filename, int num_joints) {
    FILE* fp = fopen(filename, "r");
    if (!fp) {
        utils_log_error("Cannot open input file");
        return NULL;
    }

    MotionData* data = (MotionData*)malloc(sizeof(MotionData));
    if (!data) { fclose(fp); return NULL; }
    data->num_joints  = num_joints;
    data->num_frames  = 0;
    data->frames      = (double**)malloc(MAX_FRAMES * sizeof(double*));
    if (!data->frames) { free(data); fclose(fp); return NULL; }

    char line[MAX_LINE_LENGTH];
    int is_header = 1;     /* 1 = we have not yet decided whether row 0 is a header */

    while (fgets(line, MAX_LINE_LENGTH, fp) && data->num_frames < MAX_FRAMES) {

        /* ── Header detection (run only once) ─────────────────────────── */
        if (is_header) {
            is_header = 0;
            char first = line[0];
            /* If the first character is NOT a digit, minus, or decimal point
               treat this row as a text header and skip it entirely.          */
            if ((first < '0' || first > '9') && first != '-' && first != '.') {
                continue;   /* skip the header row */
            }
            /* Otherwise fall through and parse this row as data */
        }

        /* ── Parse exactly (num_joints * 3) numeric columns ───────────── *
         *  CSV layout: joint0_x, joint0_y, joint0_z, joint1_x, ...        *
         *  There is NO leading time/index column in this dataset.          */
        double* frame = (double*)malloc(num_joints * 3 * sizeof(double));
        if (!frame) continue;

        int   col   = 0;
        char* token = strtok(line, ",\r\n");

        while (token && col < num_joints * 3) {
            frame[col++] = atof(token);
            token = strtok(NULL, ",\r\n");
        }

        if (col == num_joints * 3) {
            data->frames[data->num_frames++] = frame;
        } else {
            /* Row had wrong column count — discard silently */
            free(frame);
        }
    }

    fclose(fp);

    if (data->num_frames == 0) {
        utils_log_error("No valid frames loaded — check CSV format");
        free(data->frames);
        free(data);
        return NULL;
    }

    printf("[INFO] Loaded %d frames for %d joints\n", data->num_frames, num_joints);
    return data;
}

void io_free_motion_data(MotionData* data) {
    if (!data) return;
    for (int i = 0; i < data->num_frames; i++) free(data->frames[i]);
    free(data->frames);
    free(data);
}

FILE* io_open_output_csv(const char* filename, int num_joints, int state_dim) {
    FILE* fp = fopen(filename, "w");
    if (!fp) { utils_log_error("Cannot create output file"); return NULL; }

    /* Header: frame, then 12 columns per joint */
    fprintf(fp, "frame");
    for (int j = 0; j < num_joints; j++) {
        fprintf(fp, ",j%d_px,j%d_vx,j%d_ax,j%d_jx", j, j, j, j);
        fprintf(fp, ",j%d_py,j%d_vy,j%d_ay,j%d_jy", j, j, j, j);
        fprintf(fp, ",j%d_pz,j%d_vz,j%d_az,j%d_jz", j, j, j, j);
    }
    fprintf(fp, "\n");
    return fp;
}

void io_write_state_row(FILE* fp, int frame_idx, const double* state, int total_dim) {
    fprintf(fp, "%d", frame_idx);
    for (int i = 0; i < total_dim; i++) fprintf(fp, ",%.8f", state[i]);
    fprintf(fp, "\n");
}

void io_close_csv(FILE* fp) { if (fp) fclose(fp); }

In [ ]:
%%writefile simd_utils.h
#ifndef SIMD_UTILS_H
#define SIMD_UTILS_H

#include <immintrin.h>

__m256d simd_add_4d(__m256d a, __m256d b);
__m256d simd_mul_4d(__m256d a, __m256d b);
double simd_dot_4d(__m256d a, __m256d b);

#endif


In [ ]:
%%writefile simd_utils.c
#include "simd_utils.h"

__m256d simd_add_4d(__m256d a, __m256d b) {
    return _mm256_add_pd(a, b);
}

__m256d simd_mul_4d(__m256d a, __m256d b) {
    return _mm256_mul_pd(a, b);
}

double simd_dot_4d(__m256d a, __m256d b) {
    // Multiply
    __m256d mul = _mm256_mul_pd(a, b);

    // Horizontal add requires shuffling to sum elements within the register
    __m256d t1 = _mm256_hadd_pd(mul, mul);
    __m128d t2 = _mm256_extractf128_pd(t1, 1);
    __m128d t3 = _mm256_castpd256_pd128(t1);
    __m128d sum = _mm_add_pd(t2, t3);

    return _mm_cvtsd_f64(sum);
}

In [ ]:
%%writefile utils.h
#ifndef UTILS_H
#define UTILS_H

#include <stdlib.h>

double* utils_alloc_aligned(int rows, int cols);
void utils_free_aligned(double* ptr);
void utils_log_error(const char* msg);

#endif


In [ ]:
%%writefile utils.c
#include "utils.h"
#include <stdio.h>
#include <string.h>

double* utils_alloc_aligned(int rows, int cols) {
    double* ptr = NULL;
    if (posix_memalign((void**)&ptr, 64, rows * cols * sizeof(double)) != 0) {
        utils_log_error("Memory allocation failed.");
        return NULL;
    }
    memset(ptr, 0, rows * cols * sizeof(double));
    return ptr;
}

void utils_free_aligned(double* ptr) {
    if (ptr) free(ptr);
}

void utils_log_error(const char* msg) {
    fprintf(stderr, "[ERROR] %s\n", msg);
}

In [ ]:
%%writefile state.h
#ifndef STATE_H
#define STATE_H

void state_init_F(double* F, double dt);
void state_init_Q(double* Q);
void state_predict_x(double* x, const double* F, int dim);
void state_predict_P(double* P, const double* F, const double* Q, int dim);

#endif


In [ ]:
%%writefile state.c
#include "state.h"
#include "matrix.h"
#include "utils.h"
#include <string.h>

/* ── State-transition matrix F (12×12 per joint) ──────────────────────────
 * State order: [px, vx, ax, jx,  py, vy, ay, jy,  pz, vz, az, jz]
 * Three independent 4×4 kinematic blocks (one per axis):
 *
 *   [ 1   dt  dt²/2  dt³/6 ]
 *   [ 0    1    dt   dt²/2 ]
 *   [ 0    0     1     dt  ]
 *   [ 0    0     0      1  ]
 *
 * dt³/6 and dt²/2 follow directly from the Taylor expansion of
 * constant-jerk motion — these coefficients are exact.               */
void state_init_F(double* F, double dt) {
    mat_eye(F, 12);
    double dt2 = 0.5  * dt * dt;
    double dt3 = (1.0 / 6.0) * dt * dt * dt;

    for (int i = 0; i < 3; i++) {
        int r = i * 4;                      /* row/col base for this axis block */
        F[ r      * 12 + r + 1] = dt;
        F[ r      * 12 + r + 2] = dt2;
        F[ r      * 12 + r + 3] = dt3;

        F[(r + 1) * 12 + r + 2] = dt;
        F[(r + 1) * 12 + r + 3] = dt2;

        F[(r + 2) * 12 + r + 3] = dt;
    }
}

/* ── Process-noise covariance Q (12×12 per joint) ─────────────────────────
 * Diagonal; values chosen so position noise << velocity << accel ≈ jerk.
 * Keeping jerk noise at 1e-4 (not 1e-2) prevents filter over-excitement
 * and keeps estimated trajectories smooth for typical walking speeds.   */
void state_init_Q(double* Q) {
    mat_eye(Q, 12);
    for (int i = 0; i < 12; i++) {
        switch (i % 4) {
            case 0: Q[i * 12 + i] = 1e-6; break;   /* position   */
            case 1: Q[i * 12 + i] = 1e-5; break;   /* velocity   */
            case 2: Q[i * 12 + i] = 1e-4; break;   /* accel      */
            case 3: Q[i * 12 + i] = 1e-4; break;   /* jerk       */
        }
    }
}

/* ── Predict state: x ← F·x ───────────────────────────────────────────── */
void state_predict_x(double* x, const double* F, int dim) {
    double* next_x = utils_alloc_aligned(dim, 1);
    if (!next_x) return;
    mat_mul(F, x, next_x, dim, dim, 1);
    memcpy(x, next_x, (size_t)dim * sizeof(double));
    utils_free_aligned(next_x);
}

/* ── Predict covariance: P ← F·P·F^T + Q ─────────────────────────────── *
 * Two temporaries used so no aliased reads/writes in mat_mul calls.      */
void state_predict_P(double* P, const double* F, const double* Q, int dim) {
    double* Ft   = utils_alloc_aligned(dim, dim);
    double* tmpP = utils_alloc_aligned(dim, dim);
    if (!Ft || !tmpP) { utils_free_aligned(Ft); utils_free_aligned(tmpP); return; }

    mat_transpose(F, Ft, dim, dim);          /* Ft   = F^T           */
    mat_mul(F,    P,  tmpP, dim, dim, dim);  /* tmpP = F·P           */
    mat_mul(tmpP, Ft, P,    dim, dim, dim);  /* P    = F·P·F^T       */
    mat_add(P,    Q,  P,    dim, dim);       /* P    = F·P·F^T + Q   */

    utils_free_aligned(Ft);
    utils_free_aligned(tmpP);
}

In [ ]:
%%writefile measurement.h
#ifndef MEASUREMENT_H
#define MEASUREMENT_H

void meas_init_H(double* H);
void meas_init_R(double* R);

#endif


In [ ]:
%%writefile measurement.c
#include "measurement.h"
#include "matrix.h"
#include <string.h>

void meas_init_H(double* H) {
    // 3 rows, 12 cols. Extracts Px, Py, Pz (Indices 0, 4, 8)
    memset(H, 0, 3 * 12 * sizeof(double));
    H[0*12 + 0] = 1.0;
    H[1*12 + 4] = 1.0;
    H[2*12 + 8] = 1.0;
}

void meas_init_R(double* R) {
    // 3x3 matrix using the provided variance constants
    const double EST_R_PX = 0.29472279;
    const double EST_R_PY = 0.09632091;
    const double EST_R_PZ = 0.00204269;

    memset(R, 0, 3 * 3 * sizeof(double));
    R[0*3 + 0] = EST_R_PX;
    R[1*3 + 1] = EST_R_PY;
    R[2*3 + 2] = EST_R_PZ;
}

In [ ]:
%%writefile matrix.h
#ifndef MATRIX_H
#define MATRIX_H

#include <stdlib.h>

double* mat_alloc(int rows, int cols);
void mat_free(double* mat);
void mat_eye(double* M, int n);
void mat_mul(const double* A, const double* B, double* C, int n, int m, int p);
void mat_add(const double* A, const double* B, double* C, int r, int c);
void mat_sub(const double* A, const double* B, double* C, int r, int c);
void mat_transpose(const double* A, double* At, int r, int c);
void mat_joseph_update(double* P, const double* K, const double* H, const double* R, int n, int m);
int mat_inverse_3x3(const double* m, double* inv);

#endif


In [ ]:
%%writefile matrix.c
#include "matrix.h"
#include <stdio.h>
#include <math.h>
#include <immintrin.h>
#include <string.h>

double* mat_alloc(int rows, int cols) {
    double* ptr = NULL;
    /* 64-byte alignment: AVX-512 ready, prevents SIMD segfaults */
    if (posix_memalign((void**)&ptr, 64, (size_t)rows * cols * sizeof(double)) != 0)
        return NULL;
    memset(ptr, 0, (size_t)rows * cols * sizeof(double));
    return ptr;
}

void mat_free(double* mat) { if (mat) free(mat); }

void mat_eye(double* M, int n) {
    memset(M, 0, (size_t)n * n * sizeof(double));
    for (int i = 0; i < n; i++) M[i * n + i] = 1.0;
}

/* ── IKJ Matrix Multiplication with AVX2 FMA ──────────────────────────────
 * Multiplies A (n×m) × B (m×p) → C (n×p).
 * IKJ loop order keeps B's inner scan sequential → cache-friendly.
 * _mm256_fmadd_pd fuses multiply-add: fewer instructions, better throughput. */
void mat_mul(const double* A, const double* B, double* C, int n, int m, int p) {
    memset(C, 0, (size_t)n * p * sizeof(double));
    for (int i = 0; i < n; i++) {
        for (int k = 0; k < m; k++) {
            __m256d va = _mm256_set1_pd(A[i * m + k]);
            int j = 0;
            for (; j <= p - 4; j += 4) {
                __m256d vb = _mm256_loadu_pd(&B[k * p + j]);
                __m256d vc = _mm256_loadu_pd(&C[i * p + j]);
                _mm256_storeu_pd(&C[i * p + j], _mm256_fmadd_pd(va, vb, vc));
            }
            /* Scalar tail for columns not divisible by 4 */
            for (; j < p; j++) C[i * p + j] += A[i * m + k] * B[k * p + j];
        }
    }
}

void mat_add(const double* A, const double* B, double* C, int r, int c) {
    int total = r * c, i = 0;
    for (; i <= total - 4; i += 4) {
        __m256d va = _mm256_loadu_pd(&A[i]);
        __m256d vb = _mm256_loadu_pd(&B[i]);
        _mm256_storeu_pd(&C[i], _mm256_add_pd(va, vb));
    }
    for (; i < total; i++) C[i] = A[i] + B[i];
}

void mat_sub(const double* A, const double* B, double* C, int r, int c) {
    int total = r * c, i = 0;
    for (; i <= total - 4; i += 4) {
        __m256d va = _mm256_loadu_pd(&A[i]);
        __m256d vb = _mm256_loadu_pd(&B[i]);
        _mm256_storeu_pd(&C[i], _mm256_sub_pd(va, vb));
    }
    for (; i < total; i++) C[i] = A[i] - B[i];
}

void mat_transpose(const double* A, double* At, int r, int c) {
    for (int i = 0; i < r; i++)
        for (int j = 0; j < c; j++)
            At[j * r + i] = A[i * c + j];
}

/* ── Joseph Form covariance update ─────────────────────────────────────────
 * P = (I - K*H) * P * (I - K*H)^T  +  K * R * K^T
 *
 * Shapes:  K is (n×m),  H is (m×n),  P is (n×n),  R is (m×m)
 *
 * BUG-FIX 1: ImKHt must be (n×n), NOT (n×m).  ImKH = I - K*H is n×n,
 *            so its transpose is also n×n.  Previously mat_alloc(n,m)
 *            only allocated 36 doubles for a 144-element matrix → heap
 *            corruption on every update step.
 *
 * BUG-FIX 2: Kt (transpose of K) is (m×n), NOT (n×m).  K is n×m so
 *            Kt is m×n.  Previously mat_alloc(n,m) had wrong row/col
 *            order, causing mat_mul(KR, Kt, KRKt, n, m, n) to read
 *            out-of-bounds memory.
 */
void mat_joseph_update(double* P, const double* K, const double* H,
                       const double* R, int n, int m) {
    /* Allocations — all heap, sizes verified against their mathematical roles */
    double* I_mat = mat_alloc(n, n);   /* n×n identity                        */
    double* KH    = mat_alloc(n, n);   /* K(n×m) × H(m×n) → n×n              */
    double* ImKH  = mat_alloc(n, n);   /* I - KH  → n×n                       */
    double* ImKHt = mat_alloc(n, n);   /* (I-KH)^T → n×n  ← FIX 1 was (n,m) */
    double* T1    = mat_alloc(n, n);   /* (I-KH)×P → n×n                      */
    double* T2    = mat_alloc(n, n);   /* T1×(I-KH)^T → n×n                   */
    double* Kt    = mat_alloc(m, n);   /* K^T → m×n       ← FIX 2 was (n,m)  */
    double* KR    = mat_alloc(n, m);   /* K(n×m)×R(m×m) → n×m                 */
    double* KRKt  = mat_alloc(n, n);   /* KR(n×m)×Kt(m×n) → n×n               */

    mat_eye(I_mat, n);

    mat_mul(K,    H,     KH,    n, m, n);   /* KH    = K × H    */
    mat_sub(I_mat, KH,   ImKH,  n, n);      /* ImKH  = I - KH   */
    mat_transpose(ImKH,  ImKHt, n, n);      /* ImKHt = (I-KH)^T */

    mat_mul(ImKH, P,     T1,    n, n, n);   /* T1    = (I-KH)×P            */
    mat_mul(T1,   ImKHt, T2,    n, n, n);   /* T2    = (I-KH)×P×(I-KH)^T  */

    mat_transpose(K,  Kt, n, m);            /* Kt    = K^T (m×n)           */
    mat_mul(K,  R,  KR,   n, m, m);         /* KR    = K×R                 */
    mat_mul(KR, Kt, KRKt, n, m, n);         /* KRKt  = K×R×K^T             */

    mat_add(T2, KRKt, P, n, n);             /* P     = T2 + KRKt           */

    mat_free(I_mat); mat_free(KH);   mat_free(ImKH);  mat_free(ImKHt);
    mat_free(T1);    mat_free(T2);   mat_free(Kt);    mat_free(KR);
    mat_free(KRKt);
}

/* ── Analytic 3×3 inverse via Cramer's Rule ─────────────────────────────
 * Returns 1 on success, 0 if matrix is singular (|det| < 1e-18).
 * Used for the innovation covariance S = H×P×H^T + R (always 3×3).     */
int mat_inverse_3x3(const double* m, double* inv) {
    double det = m[0] * (m[4]*m[8] - m[5]*m[7])
               - m[1] * (m[3]*m[8] - m[5]*m[6])
               + m[2] * (m[3]*m[7] - m[4]*m[6]);

    if (fabs(det) < 1e-18) return 0;   /* singular — skip update */

    double id = 1.0 / det;
    inv[0] =  (m[4]*m[8] - m[5]*m[7]) * id;
    inv[1] =  (m[2]*m[7] - m[1]*m[8]) * id;
    inv[2] =  (m[1]*m[5] - m[2]*m[4]) * id;
    inv[3] =  (m[5]*m[6] - m[3]*m[8]) * id;
    inv[4] =  (m[0]*m[8] - m[2]*m[6]) * id;
    inv[5] =  (m[2]*m[3] - m[0]*m[5]) * id;
    inv[6] =  (m[3]*m[7] - m[4]*m[6]) * id;
    inv[7] =  (m[1]*m[6] - m[0]*m[7]) * id;
    inv[8] =  (m[0]*m[4] - m[1]*m[3]) * id;
    return 1;
}

In [ ]:
%%writefile atan_utils.h
#ifndef ATAN_UTILS_H
#define ATAN_UTILS_H

#define PI_VAL 3.14159265358979323846
#define PI_2   1.57079632679489661923

double fast_atan2(double y, double x);

#endif


In [ ]:
%%writefile atan_utils.c
#include "atan_utils.h"
#include <math.h> // Only for fabs() - we can replace with (y < 0 ? -y : y)

double fast_atan2(double y, double x) {
    // Handle the origin case
    if (x == 0.0 && y == 0.0) return 0.0;

    double abs_y = (y < 0) ? -y : y;
    double abs_x = (x < 0) ? -x : x;
    double angle;

    // Optimization: We compute atan(z) where z is in [0, 1]
    if (abs_x >= abs_y) {
        double z = abs_y / abs_x;
        // Fast approximation for atan(z) in [0, 1]:
        // atan(z) ≈ (PI/4)z - z(|z| - 1)(0.2447 + 0.0663|z|)
        angle = (PI_VAL/4.0) * z - z * (z - 1.0) * (0.2447 + 0.0663 * z);
    } else {
        double z = abs_x / abs_y;
        angle = PI_2 - ((PI_VAL/4.0) * z - z * (z - 1.0) * (0.2447 + 0.0663 * z));
    }

    // Quadrant adjustments
    if (x < 0) {
        if (y >= 0) angle = PI_VAL - angle;
        else angle = angle - PI_VAL;
    } else {
        if (y < 0) angle = -angle;
    }

    return angle;
}

In [ ]:
%%writefile test_bases.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <string.h>

#include "matrix.h"
#include "state.h"
#include "measurement.h"
#include "atan_utils.h"
#include "utils.h"
#include "simd_utils.h"

#define TOLERANCE 1e-9
#define ATAN_TOLERANCE 0.005  // ~0.3 degrees

// ============================================================================
// TEST UTILITIES
// ============================================================================
static int tests_passed = 0;
static int tests_failed = 0;

void print_matrix(const char* name, const double* M, int r, int c) {
    printf("\n%s (%dx%d):\n", name, r, c);
    for (int i = 0; i < r; i++) {
        printf("  [");
        for (int j = 0; j < c; j++) {
            printf("%10.6f", M[i * c + j]);
            if (j < c - 1) printf(", ");
        }
        printf("]\n");
    }
}

int check_close(double a, double b, double tol) {
    return fabs(a - b) < tol;
}

void test_result(const char* name, int passed) {
    if (passed) {
        printf("[PASS] %s\n", name);
        tests_passed++;
    } else {
        printf("[FAIL] %s\n", name);
        tests_failed++;
    }
}

// ============================================================================
// TEST 1: Matrix Identity
// ============================================================================
void test_mat_eye() {
    double* I = mat_alloc(4, 4);
    mat_eye(I, 4);

    int passed = 1;
    for (int i = 0; i < 4; i++) {
        for (int j = 0; j < 4; j++) {
            double expected = (i == j) ? 1.0 : 0.0;
            if (!check_close(I[i*4 + j], expected, TOLERANCE)) {
                passed = 0;
            }
        }
    }
    test_result("mat_eye creates correct identity matrix", passed);
    mat_free(I);
}

// ============================================================================
// TEST 2: Matrix Multiplication
// ============================================================================
void test_mat_mul() {
    // Test: [1,2; 3,4] * [5,6; 7,8] = [19,22; 43,50]
    double A[4] = {1, 2, 3, 4};
    double B[4] = {5, 6, 7, 8};
    double C[4];

    mat_mul(A, B, C, 2, 2, 2);

    int passed = check_close(C[0], 19, TOLERANCE) &&
                 check_close(C[1], 22, TOLERANCE) &&
                 check_close(C[2], 43, TOLERANCE) &&
                 check_close(C[3], 50, TOLERANCE);

    test_result("mat_mul 2x2 multiplication", passed);

    // Test with identity: A * I = A
    double* I = mat_alloc(2, 2);
    double* R = mat_alloc(2, 2);
    mat_eye(I, 2);
    mat_mul(A, I, R, 2, 2, 2);

    passed = check_close(R[0], A[0], TOLERANCE) &&
             check_close(R[1], A[1], TOLERANCE) &&
             check_close(R[2], A[2], TOLERANCE) &&
             check_close(R[3], A[3], TOLERANCE);

    test_result("mat_mul A*I = A property", passed);

    mat_free(I);
    mat_free(R);
}

// ============================================================================
// TEST 3: Matrix Transpose
// ============================================================================
void test_mat_transpose() {
    double A[6] = {1, 2, 3, 4, 5, 6};  // 2x3
    double At[6];  // 3x2

    mat_transpose(A, At, 2, 3);

    // Expected: [1,4; 2,5; 3,6]
    int passed = check_close(At[0], 1, TOLERANCE) &&
                 check_close(At[1], 4, TOLERANCE) &&
                 check_close(At[2], 2, TOLERANCE) &&
                 check_close(At[3], 5, TOLERANCE) &&
                 check_close(At[4], 3, TOLERANCE) &&
                 check_close(At[5], 6, TOLERANCE);

    test_result("mat_transpose 2x3 -> 3x2", passed);
}

// ============================================================================
// TEST 4: 3x3 Matrix Inverse
// ============================================================================
void test_mat_inverse_3x3() {
    // Test with a known invertible matrix
    double A[9] = {1, 2, 3,
                   0, 1, 4,
                   5, 6, 0};
    double Ainv[9];
    double I_check[9];

    int success = mat_inverse_3x3(A, Ainv);
    test_result("mat_inverse_3x3 returns success for invertible matrix", success == 1);

    // Verify A * A^-1 = I
    mat_mul(A, Ainv, I_check, 3, 3, 3);

    int passed = 1;
    for (int i = 0; i < 3; i++) {
        for (int j = 0; j < 3; j++) {
            double expected = (i == j) ? 1.0 : 0.0;
            if (!check_close(I_check[i*3 + j], expected, 1e-6)) {
                passed = 0;
            }
        }
    }
    test_result("mat_inverse_3x3: A * A^-1 = I", passed);

    // Test singular matrix
    double singular[9] = {1, 2, 3, 2, 4, 6, 1, 1, 1};
    success = mat_inverse_3x3(singular, Ainv);
    test_result("mat_inverse_3x3 returns 0 for singular matrix", success == 0);
}

// ============================================================================
// TEST 5: State Transition Matrix F Structure
// ============================================================================
void test_F_matrix_structure() {
    double* F = mat_alloc(12, 12);
    double dt = 0.033;  // ~30 FPS
    state_init_F(F, dt);

    double dt2 = 0.5 * dt * dt;
    double dt3 = (1.0 / 6.0) * dt * dt * dt;

    int passed = 1;

    // Check X-axis block (rows 0-3, cols 0-3)
    passed &= check_close(F[0*12 + 0], 1.0, TOLERANCE);      // p_x -> p_x
    passed &= check_close(F[0*12 + 1], dt, TOLERANCE);       // v_x -> p_x
    passed &= check_close(F[0*12 + 2], dt2, TOLERANCE);      // a_x -> p_x
    passed &= check_close(F[0*12 + 3], dt3, TOLERANCE);      // j_x -> p_x
    passed &= check_close(F[1*12 + 2], dt, TOLERANCE);       // a_x -> v_x
    passed &= check_close(F[1*12 + 3], dt2, TOLERANCE);      // j_x -> v_x
    passed &= check_close(F[2*12 + 3], dt, TOLERANCE);       // j_x -> a_x

    test_result("F matrix X-axis block structure", passed);

    // Check that cross-axis terms are zero
    passed = 1;
    passed &= check_close(F[0*12 + 4], 0.0, TOLERANCE);  // X doesn't affect Y
    passed &= check_close(F[0*12 + 8], 0.0, TOLERANCE);  // X doesn't affect Z
    passed &= check_close(F[4*12 + 0], 0.0, TOLERANCE);  // Y doesn't affect X

    test_result("F matrix block-diagonal (no cross-axis coupling)", passed);

    print_matrix("F matrix (dt=0.033)", F, 12, 12);
    mat_free(F);
}

// ============================================================================
// TEST 6: Measurement Matrix H Structure
// ============================================================================
void test_H_matrix_structure() {
    double H[3 * 12];
    meas_init_H(H);

    int passed = 1;

    // H should extract p_x (idx 0), p_y (idx 4), p_z (idx 8)
    passed &= check_close(H[0*12 + 0], 1.0, TOLERANCE);  // Row 0 picks p_x
    passed &= check_close(H[1*12 + 4], 1.0, TOLERANCE);  // Row 1 picks p_y
    passed &= check_close(H[2*12 + 8], 1.0, TOLERANCE);  // Row 2 picks p_z

    // All other entries should be zero
    for (int i = 0; i < 3; i++) {
        for (int j = 0; j < 12; j++) {
            if ((i == 0 && j == 0) || (i == 1 && j == 4) || (i == 2 && j == 8))
                continue;
            if (!check_close(H[i*12 + j], 0.0, TOLERANCE)) {
                passed = 0;
            }
        }
    }

    test_result("H matrix extracts positions correctly", passed);
    print_matrix("H matrix", H, 3, 12);
}

// ============================================================================
// TEST 7: State Prediction Sanity Check
// ============================================================================
void test_state_prediction() {
    double* F = mat_alloc(12, 12);
    double dt = 0.1;
    state_init_F(F, dt);

    // Initial state: position=1, velocity=2, accel=0, jerk=0 for all axes
    double x[12] = {1.0, 2.0, 0.0, 0.0,   // X: p,v,a,j
                    1.0, 2.0, 0.0, 0.0,   // Y
                    1.0, 2.0, 0.0, 0.0};  // Z

    state_predict_x(x, F, 12);

    // Expected: p_new = p + v*dt = 1 + 2*0.1 = 1.2
    int passed = check_close(x[0], 1.2, 1e-6) &&
                 check_close(x[4], 1.2, 1e-6) &&
                 check_close(x[8], 1.2, 1e-6);

    test_result("state_predict_x: position advances correctly", passed);

    // Velocity should remain 2.0 (no acceleration)
    passed = check_close(x[1], 2.0, 1e-6) &&
             check_close(x[5], 2.0, 1e-6) &&
             check_close(x[9], 2.0, 1e-6);

    test_result("state_predict_x: velocity unchanged when a=0", passed);

    mat_free(F);
}

// ============================================================================
// TEST 8: Fast atan2 Accuracy
// ============================================================================
void test_fast_atan2() {
    double max_error = 0.0;
    int passed = 1;

    // Test all quadrants and edge cases
    double test_cases[][2] = {
        {1.0, 1.0},   {1.0, -1.0},  {-1.0, 1.0},  {-1.0, -1.0},  // Quadrants
        {0.0, 1.0},   {1.0, 0.0},   {0.0, -1.0},  {-1.0, 0.0},   // Axes
        {0.5, 2.0},   {2.0, 0.5},   {-0.5, 2.0},  {-2.0, -0.5},  // Various ratios
        {0.001, 1.0}, {1.0, 0.001}, {100.0, 1.0}, {1.0, 100.0},  // Extreme ratios
    };

    int n_cases = sizeof(test_cases) / sizeof(test_cases[0]);

    for (int i = 0; i < n_cases; i++) {
        double y = test_cases[i][0];
        double x = test_cases[i][1];

        double fast = fast_atan2(y, x);
        double ref = atan2(y, x);
        double error = fabs(fast - ref);

        if (error > max_error) max_error = error;
        if (error > ATAN_TOLERANCE) {
            printf("  fast_atan2(%.3f, %.3f) = %.6f, expected %.6f, error = %.6f\n",
                   y, x, fast, ref, error);
            passed = 0;
        }
    }

    printf("  Max fast_atan2 error: %.6f rad (%.3f deg)\n", max_error, max_error * 180 / PI_VAL);
    test_result("fast_atan2 accuracy within tolerance", passed);

    // Special case: origin
    double origin = fast_atan2(0.0, 0.0);
    test_result("fast_atan2(0,0) = 0", check_close(origin, 0.0, TOLERANCE));
}

// ============================================================================
// TEST 9: Covariance Prediction P stays positive semi-definite
// ============================================================================
void test_covariance_prediction() {
    double* F = mat_alloc(12, 12);
    double* Q = mat_alloc(12, 12);
    double* P = mat_alloc(12, 12);

    state_init_F(F, 0.033);
    state_init_Q(Q);
    mat_eye(P, 12);  // Start with identity covariance

    // Run multiple prediction steps
    for (int step = 0; step < 100; step++) {
        state_predict_P(P, F, Q, 12);
    }

    // Check diagonal elements are positive (necessary for PSD)
    int passed = 1;
    for (int i = 0; i < 12; i++) {
        if (P[i*12 + i] <= 0) {
            printf("  P[%d,%d] = %.6e (should be positive)\n", i, i, P[i*12 + i]);
            passed = 0;
        }
    }
    test_result("P matrix diagonals remain positive after 100 predictions", passed);

    // Check symmetry
    passed = 1;
    for (int i = 0; i < 12; i++) {
        for (int j = i+1; j < 12; j++) {
            if (!check_close(P[i*12 + j], P[j*12 + i], 1e-10)) {
                passed = 0;
            }
        }
    }
    test_result("P matrix remains symmetric", passed);

    mat_free(F);
    mat_free(Q);
    mat_free(P);
}

// ============================================================================
// TEST 10: Full LKF Update Cycle (One Step)
// ============================================================================
void test_lkf_single_update() {
    const int n = 12;  // State dim
    const int m = 3;   // Measurement dim

    double* F = mat_alloc(n, n);
    double* Q = mat_alloc(n, n);
    double* P = mat_alloc(n, n);
    double* H = mat_alloc(m, n);
    double* R = mat_alloc(m, m);
    double* x = mat_alloc(n, 1);

    state_init_F(F, 0.033);
    state_init_Q(Q);
    meas_init_H(H);
    meas_init_R(R);
    mat_eye(P, n);

    // Initial state: position at origin, moving in +X direction
    x[0] = 0.0; x[1] = 1.0; x[2] = 0.0; x[3] = 0.0;  // X
    x[4] = 0.0; x[5] = 0.0; x[6] = 0.0; x[7] = 0.0;  // Y
    x[8] = 0.0; x[9] = 0.0; x[10] = 0.0; x[11] = 0.0; // Z

    // Prediction step
    state_predict_x(x, F, n);
    state_predict_P(P, F, Q, n);

    // Measurement: observe position at (0.04, 0.01, 0.0)
    double z[3] = {0.04, 0.01, 0.0};

    // Compute innovation: y = z - H*x
    double Hx[3];
    mat_mul(H, x, Hx, m, n, 1);
    double y[3] = {z[0] - Hx[0], z[1] - Hx[1], z[2] - Hx[2]};

    // Innovation covariance: S = H*P*H' + R
    double* PHt = mat_alloc(n, m);
    double* Ht = mat_alloc(n, m);
    double* S = mat_alloc(m, m);
    double* HPHt = mat_alloc(m, m);

    mat_transpose(H, Ht, m, n);
    mat_mul(P, Ht, PHt, n, n, m);
    mat_mul(H, PHt, HPHt, m, n, m);
    mat_add(HPHt, R, S, m, m);

    // Kalman gain: K = P*H' * S^-1
    double Sinv[9];
    int inv_ok = mat_inverse_3x3(S, Sinv);
    test_result("Innovation covariance S is invertible", inv_ok == 1);

    double* K = mat_alloc(n, m);
    mat_mul(PHt, Sinv, K, n, m, m);

    // State update: x = x + K*y
    double Ky[12];
    mat_mul(K, y, Ky, n, m, 1);
    for (int i = 0; i < n; i++) x[i] += Ky[i];

    // Covariance update (Joseph form)
    mat_joseph_update(P, K, H, R, n, m);

    // Check: estimate should be between prediction and measurement
    int passed = (x[0] > 0.033 - 0.01 && x[0] < 0.04 + 0.01);
    test_result("LKF update: position estimate is reasonable", passed);

    // Check P still positive definite
    passed = 1;
    for (int i = 0; i < n; i++) {
        if (P[i*n + i] <= 0) passed = 0;
    }
    test_result("LKF update: P remains positive definite", passed);

    printf("\n  After LKF update:\n");
    printf("    x_position = [%.6f, %.6f, %.6f]\n", x[0], x[4], x[8]);
    printf("    measurement = [%.6f, %.6f, %.6f]\n", z[0], z[1], z[2]);

    mat_free(F); mat_free(Q); mat_free(P); mat_free(H); mat_free(R);
    mat_free(x); mat_free(PHt); mat_free(Ht); mat_free(S); mat_free(HPHt);
    mat_free(K);
}

// ============================================================================
// TEST 11: SIMD Operations
// ============================================================================
void test_simd_operations() {
    double a_data[4] __attribute__((aligned(32))) = {1.0, 2.0, 3.0, 4.0};
    double b_data[4] __attribute__((aligned(32))) = {5.0, 6.0, 7.0, 8.0};

    __m256d a = _mm256_load_pd(a_data);
    __m256d b = _mm256_load_pd(b_data);

    double dot = simd_dot_4d(a, b);
    double expected = 1*5 + 2*6 + 3*7 + 4*8;  // 70

    test_result("simd_dot_4d computes correct dot product", check_close(dot, expected, TOLERANCE));
}

// ============================================================================
// MAIN
// ============================================================================
int main() {
    printf("\n========================================\n");
    printf("  KALMAN FILTER BASE MODULE TESTS\n");
    printf("========================================\n\n");

    test_mat_eye();
    test_mat_mul();
    test_mat_transpose();
    test_mat_inverse_3x3();
    test_F_matrix_structure();
    test_H_matrix_structure();
    test_state_prediction();
    test_fast_atan2();
    test_covariance_prediction();
    test_lkf_single_update();
    test_simd_operations();

    printf("\n========================================\n");
    printf("  RESULTS: %d passed, %d failed\n", tests_passed, tests_failed);
    printf("========================================\n");

    return tests_failed > 0 ? 1 : 0;
}


In [ ]:
# Compile tests
!gcc -O3 -march=native -mavx2 -mfma -o test_bases \
    test_bases.c matrix.c state.c measurement.c atan_utils.c utils.c simd_utils.c \
    -lm

# Run tests
!./test_bases


In [ ]:
%%writefile lkf.h
#ifndef LKF_H
#define LKF_H

#define NUM_JOINTS 23
#define STATE_DIM 12     // per joint: [px,vx,ax,jx, py,vy,ay,jy, pz,vz,az,jz]
#define MEAS_DIM 3       // per joint: [px, py, pz]
#define TOTAL_STATE_DIM (NUM_JOINTS * STATE_DIM)

typedef struct {
    double* x;      // State vector [STATE_DIM]
    double* P;      // Covariance [STATE_DIM x STATE_DIM]
} JointState;

typedef struct {
    JointState joints[NUM_JOINTS];
    double* F;      // State transition [STATE_DIM x STATE_DIM]
    double* Q;      // Process noise [STATE_DIM x STATE_DIM]
    double* H;      // Measurement matrix [MEAS_DIM x STATE_DIM]
    double* R;      // Measurement noise [MEAS_DIM x MEAS_DIM]
    double dt;
} LKF;

// Lifecycle
LKF* lkf_create(double dt);
void lkf_destroy(LKF* lkf);

// Core operations
void lkf_predict(LKF* lkf);
void lkf_update(LKF* lkf, const double* measurements);  // measurements: [NUM_JOINTS * 3]

// Utilities
void lkf_get_positions(const LKF* lkf, double* positions);  // Output: [NUM_JOINTS * 3]
void lkf_get_full_state(const LKF* lkf, double* state);     // Output: [TOTAL_STATE_DIM]
void lkf_set_initial_state(LKF* lkf, int joint_idx, const double* pos);

#endif


In [ ]:
%%writefile lkf.c
#include "lkf.h"
#include "matrix.h"
#include "state.h"
#include "measurement.h"
#include "utils.h"
#include <stdio.h>
#include <string.h>

// ============================================================================
// LKF LIFECYCLE
// ============================================================================

LKF* lkf_create(double dt) {
    LKF* lkf = (LKF*)malloc(sizeof(LKF));
    if (!lkf) {
        utils_log_error("Failed to allocate LKF structure");
        return NULL;
    }

    lkf->dt = dt;

    // Allocate shared matrices
    lkf->F = mat_alloc(STATE_DIM, STATE_DIM);
    lkf->Q = mat_alloc(STATE_DIM, STATE_DIM);
    lkf->H = mat_alloc(MEAS_DIM, STATE_DIM);
    lkf->R = mat_alloc(MEAS_DIM, MEAS_DIM);

    if (!lkf->F || !lkf->Q || !lkf->H || !lkf->R) {
        utils_log_error("Failed to allocate LKF matrices");
        lkf_destroy(lkf);
        return NULL;
    }

    // Initialize matrices
    state_init_F(lkf->F, dt);
    state_init_Q(lkf->Q);
    meas_init_H(lkf->H);
    meas_init_R(lkf->R);

    // Initialize per-joint state
    for (int j = 0; j < NUM_JOINTS; j++) {
        lkf->joints[j].x = mat_alloc(STATE_DIM, 1);
        lkf->joints[j].P = mat_alloc(STATE_DIM, STATE_DIM);

        if (!lkf->joints[j].x || !lkf->joints[j].P) {
            utils_log_error("Failed to allocate joint state");
            lkf_destroy(lkf);
            return NULL;
        }

        // Initialize P as identity (moderate initial uncertainty)
        mat_eye(lkf->joints[j].P, STATE_DIM);
        // Scale initial covariance
        for (int i = 0; i < STATE_DIM; i++) {
            lkf->joints[j].P[i * STATE_DIM + i] = 1.0;
        }
    }

    return lkf;
}

void lkf_destroy(LKF* lkf) {
    if (!lkf) return;

    mat_free(lkf->F);
    mat_free(lkf->Q);
    mat_free(lkf->H);
    mat_free(lkf->R);

    for (int j = 0; j < NUM_JOINTS; j++) {
        mat_free(lkf->joints[j].x);
        mat_free(lkf->joints[j].P);
    }

    free(lkf);
}

// ============================================================================
// LKF CORE OPERATIONS
// ============================================================================

void lkf_predict(LKF* lkf) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        state_predict_x(lkf->joints[j].x, lkf->F, STATE_DIM);
        state_predict_P(lkf->joints[j].P, lkf->F, lkf->Q, STATE_DIM);
    }
}

static void lkf_update_joint(JointState* joint, const double* z,
                             const double* H, const double* R) {
    const int n = STATE_DIM;
    const int m = MEAS_DIM;

    double* x = joint->x;
    double* P = joint->P;

    // Allocate working matrices
    double* Ht = mat_alloc(n, m);
    double* PHt = mat_alloc(n, m);
    double* HPHt = mat_alloc(m, m);
    double* S = mat_alloc(m, m);
    double Sinv[9];
    double* K = mat_alloc(n, m);

    // Predicted measurement: Hx
    double Hx[3];
    mat_mul(H, x, Hx, m, n, 1);

    // Innovation: y = z - Hx
    double y[3] = {z[0] - Hx[0], z[1] - Hx[1], z[2] - Hx[2]};

    // Innovation covariance: S = H*P*H' + R
    mat_transpose(H, Ht, m, n);
    mat_mul(P, Ht, PHt, n, n, m);
    mat_mul(H, PHt, HPHt, m, n, m);
    mat_add(HPHt, R, S, m, m);

    // Invert S (3x3)
    if (!mat_inverse_3x3(S, Sinv)) {
        // Singular matrix - skip update (measurement uninformative)
        utils_log_error("Singular innovation covariance, skipping update");
        goto cleanup;
    }

    // Kalman gain: K = P*H'*S^-1
    mat_mul(PHt, Sinv, K, n, m, m);

    // State update: x = x + K*y
    double Ky[STATE_DIM];
    mat_mul(K, y, Ky, n, m, 1);
    for (int i = 0; i < n; i++) {
        x[i] += Ky[i];
    }

    // Covariance update (Joseph form for numerical stability)
    mat_joseph_update(P, K, H, R, n, m);

cleanup:
    mat_free(Ht);
    mat_free(PHt);
    mat_free(HPHt);
    mat_free(S);
    mat_free(K);
}

void lkf_update(LKF* lkf, const double* measurements) {
    // measurements is [NUM_JOINTS * 3] array: [px0,py0,pz0, px1,py1,pz1, ...]
    for (int j = 0; j < NUM_JOINTS; j++) {
        const double* z = &measurements[j * MEAS_DIM];
        lkf_update_joint(&lkf->joints[j], z, lkf->H, lkf->R);
    }
}

// ============================================================================
// UTILITIES
// ============================================================================

void lkf_set_initial_state(LKF* lkf, int joint_idx, const double* pos) {
    if (joint_idx < 0 || joint_idx >= NUM_JOINTS) return;

    double* x = lkf->joints[joint_idx].x;
    // Set position, zero out velocities/accelerations/jerks
    x[0] = pos[0]; x[1] = 0; x[2] = 0; x[3] = 0;  // X
    x[4] = pos[1]; x[5] = 0; x[6] = 0; x[7] = 0;  // Y
    x[8] = pos[2]; x[9] = 0; x[10] = 0; x[11] = 0; // Z
}

void lkf_get_positions(const LKF* lkf, double* positions) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        const double* x = lkf->joints[j].x;
        positions[j * 3 + 0] = x[0];   // px
        positions[j * 3 + 1] = x[4];   // py
        positions[j * 3 + 2] = x[8];   // pz
    }
}

void lkf_get_full_state(const LKF* lkf, double* state) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        memcpy(&state[j * STATE_DIM], lkf->joints[j].x, STATE_DIM * sizeof(double));
    }
}


In [ ]:
%%writefile lkf_main.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "lkf.h"
#include "io.h"
#include "utils.h"

int main(int argc, char* argv[]) {
    const char* input_file = (argc > 1) ? argv[1] : "gait_data.csv";
    const char* output_file = (argc > 2) ? argv[2] : "lkf_results.csv";
    double dt = (argc > 3) ? atof(argv[3]) : 0.033;  // Default ~30 FPS

    printf("========================================\n");
    printf("  LINEAR KALMAN FILTER (LKF)\n");
    printf("========================================\n");
    printf("Input:  %s\n", input_file);
    printf("Output: %s\n", output_file);
    printf("dt:     %.4f s\n", dt);
    printf("----------------------------------------\n\n");

    // Load data
    MotionData* data = io_load_csv(input_file, NUM_JOINTS);
    if (!data) {
        return 1;
    }

    // Create filter
    LKF* lkf = lkf_create(dt);
    if (!lkf) {
        io_free_motion_data(data);
        return 1;
    }

    // Initialize with first measurement
    for (int j = 0; j < NUM_JOINTS; j++) {
        double pos[3] = {
            data->frames[0][j * 3 + 0],
            data->frames[0][j * 3 + 1],
            data->frames[0][j * 3 + 2]
        };
        lkf_set_initial_state(lkf, j, pos);
    }

    // Open output file
    FILE* out_fp = io_open_output_csv(output_file, NUM_JOINTS, STATE_DIM);
    if (!out_fp) {
        lkf_destroy(lkf);
        io_free_motion_data(data);
        return 1;
    }

    // Allocate buffer for full state output
    double* full_state = (double*)malloc(TOTAL_STATE_DIM * sizeof(double));

    // Process all frames
    printf("[INFO] Processing %d frames...\n", data->num_frames);

    for (int frame = 0; frame < data->num_frames; frame++) {
        // Predict
        if (frame > 0) {
            lkf_predict(lkf);
        }

        // Update with measurement
        lkf_update(lkf, data->frames[frame]);

        // Output state
        lkf_get_full_state(lkf, full_state);
        io_write_state_row(out_fp, frame, full_state, TOTAL_STATE_DIM);

        // Progress indicator
        if ((frame + 1) % 100 == 0 || frame == data->num_frames - 1) {
            printf("\r[INFO] Processed frame %d/%d", frame + 1, data->num_frames);
            fflush(stdout);
        }
    }
    printf("\n\n");

    // Report sample results
    printf("[INFO] Sample results (Joint 0, final frame):\n");
    double* x = lkf->joints[0].x;
    printf("  Position: (%.4f, %.4f, %.4f)\n", x[0], x[4], x[8]);
    printf("  Velocity: (%.4f, %.4f, %.4f)\n", x[1], x[5], x[9]);
    printf("  Acceleration: (%.4f, %.4f, %.4f)\n", x[2], x[6], x[10]);
    printf("  Jerk: (%.4f, %.4f, %.4f)\n", x[3], x[7], x[11]);

    // Cleanup
    free(full_state);
    io_close_csv(out_fp);
    lkf_destroy(lkf);
    io_free_motion_data(data);

    printf("\n[SUCCESS] Results written to %s\n", output_file);
    return 0;
}


In [ ]:
!gcc -O3 -march=native -mavx2 -mfma -o lkf_main \
    lkf_main.c lkf.c io.c matrix.c state.c measurement.c utils.c -lm

!./lkf_main "3D Full Body Humain Gait Walking Dataset (Noisy Values).csv" "lkf_results.csv" 0.0333

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 1  —  Compile and run the LKF
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, os, sys

# ── USER SETTINGS ──────────────────────────────────────────────────────────
NOISY_CSV = "3D Full Body Humain Gait Walking Dataset (Noisy Values).csv"
TRUE_CSV  = "3D Full Body Humain Gait Walking Dataset (True Values).csv"
LKF_OUT   = "lkf_results.csv"
DT        = 1.0 / 30.0   # seconds per frame (30 FPS)
JOINT_IDX = 0             # 0 = pelvis; change to any 0–22
# ──────────────────────────────────────────────────────────────────────────

JOINT_NAMES = [
    "pelvis","L5","L3","T12","T8","neck","head",
    "shoulderRight","upperArmRight","forearmRight","handRight",
    "shoulderLeft","upperArmLeft","forearmLeft","handLeft",
    "upperLegRight","lowerLegRight","footRight","toeRight",
    "upperLegLeft","lowerLegLeft","footLeft","toeLeft",
]
JOINT_NAME = JOINT_NAMES[JOINT_IDX]

# ── Compile ────────────────────────────────────────────────────────────────
build = subprocess.run(
    ["gcc", "-O3", "-march=native", "-mavx2", "-mfma",
     "-o", "lkf_main",
     "lkf_main.c", "lkf.c", "io.c", "matrix.c",
     "state.c", "measurement.c", "utils.c", "-lm"],
    capture_output=True, text=True
)
print(build.stdout)
if build.returncode != 0:
    print(build.stderr, file=sys.stderr)
    raise RuntimeError("Build failed — check errors above")
print("✅ Compiled: lkf_main")

# ── Run LKF ───────────────────────────────────────────────────────────────
run = subprocess.run(
    ["./lkf_main", NOISY_CSV, LKF_OUT, str(DT)],
    capture_output=True, text=True
)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr, file=sys.stderr)
    raise RuntimeError("LKF runner failed")

print(f"✅ LKF results written to {LKF_OUT}")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 2  —  Load all three data sources
# ═══════════════════════════════════════════════════════════════════════════
import numpy  as np
import pandas as pd
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
from   matplotlib.lines    import Line2D
import warnings
warnings.filterwarnings("ignore")

NUM_JOINTS = 23
STATE_DIM  = 12   # per joint: [px,vx,ax,jx, py,vy,ay,jy, pz,vz,az,jz]

def load_raw_csv(path, n_joints=NUM_JOINTS):
    """Load a gait CSV. Returns ndarray (N_frames, n_joints, 3)."""
    df = pd.read_csv(path, header=None, dtype=str)
    # Drop text header row if present
    try:
        float(df.iloc[0, 0])
    except (ValueError, TypeError):
        df = df.iloc[1:].reset_index(drop=True)
    cols = n_joints * 3
    data = df.iloc[:, :cols].astype(float).values
    assert data.shape[1] == cols, f"Expected {cols} columns, got {data.shape[1]}"
    return data.reshape(-1, n_joints, 3)

noisy_arr = load_raw_csv(NOISY_CSV)   # (N, 23, 3)
true_arr  = load_raw_csv(TRUE_CSV)    # (N, 23, 3)

# Load LKF output
lkf_df = pd.read_csv(LKF_OUT)
N_LKF  = len(lkf_df)

# Extract 12-D state for the chosen joint
# Column layout: frame | j0_px j0_vx j0_ax j0_jx j0_py j0_vy ... | j1_px ...
j        = JOINT_IDX
base_col = 1 + j * STATE_DIM

def get_axis(ax_idx):
    """Returns (pos, vel, acc, jrk) for axis 0=X, 1=Y, 2=Z."""
    b = base_col + ax_idx * 4
    return (lkf_df.iloc[:, b  ].values,
            lkf_df.iloc[:, b+1].values,
            lkf_df.iloc[:, b+2].values,
            lkf_df.iloc[:, b+3].values)

px, vx, ax_, jx = get_axis(0)
py, vy, ay_, jy = get_axis(1)
pz, vz, az_, jz = get_axis(2)

N = min(N_LKF, noisy_arr.shape[0], true_arr.shape[0])
t = np.arange(N) * DT

print(f"✅ Plotting {N} frames  |  Joint {JOINT_IDX}: '{JOINT_NAME}'")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 3  —  PLOT A: 2-D Time-Series (pos / vel / acc / jerk per axis)
# ═══════════════════════════════════════════════════════════════════════════
AXES_LABELS = ["X", "Y", "Z"]
STATE_DATA  = {
    "Position (m)":        ([px[:N], py[:N], pz[:N]],    "tab:blue"),
    "Velocity (m/s)":      ([vx[:N], vy[:N], vz[:N]],    "tab:orange"),
    "Acceleration (m/s²)": ([ax_[:N], ay_[:N], az_[:N]], "tab:green"),
    "Jerk (m/s³)":         ([jx[:N], jy[:N], jz[:N]],    "tab:red"),
}

fig, axes = plt.subplots(4, 3, figsize=(16, 14), sharex=True)
fig.suptitle(
    f"LKF State Estimates — Joint {JOINT_IDX}: {JOINT_NAME}",
    fontsize=15, fontweight="bold", y=1.01
)

for row_idx, (ylabel, (series, color)) in enumerate(STATE_DATA.items()):
    for col_idx, (sig, axis_lbl) in enumerate(zip(series, AXES_LABELS)):
        ax = axes[row_idx][col_idx]
        ax.plot(t, sig, color=color, linewidth=1.2)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(True, alpha=0.3)
        if row_idx == 0:
            ax.set_title(f"{axis_lbl}-axis\n{ylabel}", fontsize=9, fontweight="bold")
        if row_idx == 3:
            ax.set_xlabel("Time (s)", fontsize=8)

plt.tight_layout()
plt.savefig("lkf_state_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: lkf_state_timeseries.png")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 4  —  PLOT B: True vs Noisy vs LKF Estimated Position
# ═══════════════════════════════════════════════════════════════════════════
true_pos  = true_arr [:N, j, :]
noisy_pos = noisy_arr[:N, j, :]
lkf_pos   = np.column_stack([px[:N], py[:N], pz[:N]])

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle(
    f"Position Comparison — Joint {JOINT_IDX}: {JOINT_NAME}",
    fontsize=14, fontweight="bold"
)

for i, lbl in enumerate(["X", "Y", "Z"]):
    ax = axes[i]
    ax.plot(t, true_pos[:, i],  color="black",    lw=1.5,             label="True",        zorder=3)
    ax.plot(t, noisy_pos[:, i], color="salmon",   lw=0.8, alpha=0.7,  label="Noisy meas.", zorder=2)
    ax.plot(t, lkf_pos[:, i],   color="tab:blue", lw=1.5, ls="--",    label="LKF estimate",zorder=4)
    ax.set_ylabel(f"{lbl} position (m)", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)", fontsize=10)
plt.tight_layout()
plt.savefig("lkf_position_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: lkf_position_comparison.png")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 5  —  PLOT C: RMSE across all 23 joints
# ═══════════════════════════════════════════════════════════════════════════
rmse_noisy = np.zeros(NUM_JOINTS)
rmse_lkf   = np.zeros(NUM_JOINTS)

for jj in range(NUM_JOINTS):
    b = 1 + jj * STATE_DIM
    lkf_j = np.column_stack([
        lkf_df.iloc[:N, b    ].values,   # px
        lkf_df.iloc[:N, b + 4].values,   # py
        lkf_df.iloc[:N, b + 8].values,   # pz
    ])
    true_j  = true_arr [:N, jj, :]
    noisy_j = noisy_arr[:N, jj, :]
    rmse_noisy[jj] = np.sqrt(np.mean((noisy_j - true_j)**2))
    rmse_lkf  [jj] = np.sqrt(np.mean((lkf_j   - true_j)**2))

x_idx = np.arange(NUM_JOINTS)
width = 0.38

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x_idx - width/2, rmse_noisy, width, color="salmon",   label="Noisy RMSE",
       alpha=0.85, edgecolor="black", linewidth=0.5)
ax.bar(x_idx + width/2, rmse_lkf,   width, color="tab:blue", label="LKF RMSE",
       alpha=0.85, edgecolor="black", linewidth=0.5)

ax.set_xticks(x_idx)
ax.set_xticklabels(JOINT_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Position RMSE (m)", fontsize=11)
ax.set_title("Position RMSE — Noisy vs LKF Estimates (all joints)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

mean_imp = (rmse_noisy.mean() - rmse_lkf.mean()) / rmse_noisy.mean() * 100
ax.text(0.98, 0.96, f"Mean RMSE reduction: {mean_imp:.1f}%",
        transform=ax.transAxes, ha="right", va="top", fontsize=11, color="darkblue",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

plt.tight_layout()
plt.savefig("lkf_rmse_all_joints.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: lkf_rmse_all_joints.png")
print(f"   Avg noisy RMSE : {rmse_noisy.mean():.4f} m")
print(f"   Avg LKF   RMSE : {rmse_lkf.mean():.4f} m")
print(f"   Improvement    : {mean_imp:.1f}%")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 6  —  PLOT D: Innovation residuals (z − Hx̂)
#  Should be zero-mean white noise if the filter is well-tuned
# ═══════════════════════════════════════════════════════════════════════════
innovation = noisy_pos - lkf_pos   # (N, 3)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
fig.suptitle(
    f"Innovation Residuals (z − H·x̂) — Joint {JOINT_IDX}: {JOINT_NAME}",
    fontsize=13, fontweight="bold"
)

for i, lbl in enumerate(["X", "Y", "Z"]):
    ax = axes[i]
    ax.plot(t, innovation[:, i], color="tab:purple", lw=0.9, alpha=0.8)
    ax.axhline(0, color="black", lw=1.2, zorder=3)
    sigma = innovation[:, i].std()
    ax.axhline( 2 * sigma, color="red", lw=1, ls="--", label="±2σ")
    ax.axhline(-2 * sigma, color="red", lw=1, ls="--")
    ax.set_ylabel(f"Residual {lbl} (m)", fontsize=9)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)", fontsize=10)
plt.tight_layout()
plt.savefig("lkf_innovations.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: lkf_innovations.png")

print("\n🎉  All four plots complete.")
print("Files: lkf_state_timeseries.png | lkf_position_comparison.png "
      "| lkf_rmse_all_joints.png | lkf_innovations.png")

In [ ]:
%%writefile ekf.h
#ifndef EKF_H
#define EKF_H

#include "lkf.h"   /* reuse NUM_JOINTS, STATE_DIM, MEAS_DIM, TOTAL_STATE_DIM */

/* ── EKF-specific numerical safety constants ───────────────────────────────
 * EPSILON_R   : minimum range r before Jacobian derivatives are undefined
 * EPSILON_RHO : minimum horizontal range ρ before azimuth/elevation
 *               derivatives hit gimbal-lock singularity
 * Both match the regularisation strategy described in Milestone 1 §7.5.1   */
#define EPSILON_R   1e-10
#define EPSILON_RHO 1e-10

/* ── Per-joint EKF state (identical layout to LKF) ────────────────────── */
typedef struct {
    double* x;   /* state vector  [STATE_DIM × 1]        */
    double* P;   /* covariance    [STATE_DIM × STATE_DIM] */
} EKFJointState;

/* ── Full EKF filter struct ────────────────────────────────────────────── *
 * Prediction matrices F and Q are shared (same constant-jerk model as LKF)*
 * R is the 3×3 measurement noise for spherical coords (σ²_r, σ²_θ, σ²_φ) *
 * H is NOT stored as a field — Hk is recomputed each update (time-varying) */
typedef struct {
    EKFJointState joints[NUM_JOINTS];
    double* F;   /* [STATE_DIM × STATE_DIM] — shared, constant              */
    double* Q;   /* [STATE_DIM × STATE_DIM] — shared, constant              */
    double* R;   /* [MEAS_DIM  × MEAS_DIM ] — spherical measurement noise   */
    double  dt;
} EKF;

/* ── Lifecycle ─────────────────────────────────────────────────────────── */
EKF*  ekf_create (double dt);
void  ekf_destroy(EKF* ekf);

/* ── Core filter operations ────────────────────────────────────────────── */
void  ekf_predict(EKF* ekf);

/* measurements: flat array [NUM_JOINTS × 3], Cartesian (px,py,pz) per joint
 * The EKF internally converts each position to spherical (r,θ,φ) and builds
 * the linearised Jacobian Hk before running the update equations.          */
void  ekf_update (EKF* ekf, const double* measurements);

/* ── Utility helpers ───────────────────────────────────────────────────── */
void  ekf_set_initial_state(EKF* ekf, int joint_idx, const double* pos);
void  ekf_get_positions    (const EKF* ekf, double* positions); /* [NUM_JOINTS×3] */
void  ekf_get_full_state   (const EKF* ekf, double* state);     /* [TOTAL_STATE_DIM] */

/* ── Internal helpers (exposed for unit-testing) ───────────────────────── */
/* Nonlinear measurement function h(x) → spherical (r, θ, φ)              */
void  ekf_compute_h       (const double* x, double* h_out);

/* Jacobian ∂h/∂x evaluated at predicted state x̂ → Hk [MEAS_DIM×STATE_DIM] */
void  ekf_compute_jacobian(const double* x, double* Hk);

/* Initialise the spherical-coordinate measurement noise matrix R          */
void  ekf_init_R          (double* R);

#endif /* EKF_H */


In [ ]:
%%writefile ekf.c
#include "ekf.h"
#include "matrix.h"
#include "state.h"
#include "atan_utils.h"
#include "utils.h"
#include <stdio.h>
#include <string.h>
#include <math.h>   /* sqrt, fabs — no atan2 from here; we use fast_atan2 */

/* ============================================================================
 * INTERNAL: Nonlinear measurement function  h(x)
 *
 * Maps the 12-D joint state to 3-D spherical coordinates:
 *
 *   h(x) = [ r ]   =  [ sqrt(px²+py²+pz²)          ]
 *           [ θ ]      [ atan2(py, px)               ]
 *           [ φ ]      [ atan2(pz, sqrt(px²+py²))   ]
 *
 * State layout: [px,vx,ax,jx, py,vy,ay,jy, pz,vz,az,jz]
 *                 0  1  2  3   4  5  6  7   8  9 10 11
 *
 * Only position components (indices 0, 4, 8) enter h(x).
 * fast_atan2 is used throughout — no <math.h> atan2 calls.
 * Ref: Milestone 1 §7.3.2 – §7.3.6
 * ========================================================================= */
void ekf_compute_h(const double* x, double* h_out) {
    double px = x[0], py = x[4], pz = x[8];

    double r = sqrt(px*px + py*py + pz*pz);
    if (r < EPSILON_R) r = EPSILON_R;   /* regularise: avoid /0 at origin */

    double rho = sqrt(px*px + py*py);   /* horizontal range */

    h_out[0] = r;
    h_out[1] = fast_atan2(py, px);                /* azimuth  θ ∈ (-π, π]  */
    h_out[2] = fast_atan2(pz, rho);               /* elevation φ ∈ [-π/2,π/2] */
}

/* ============================================================================
 * INTERNAL: Jacobian  Hk = ∂h/∂x  (3 × 12)
 *
 * Since h depends only on px(0), py(4), pz(8), exactly 9 of 36 entries
 * are non-zero (columns 1,2,3,5,6,7,9,10,11 are all zero).
 *
 * Derived quantities:
 *   r   = sqrt(px²+py²+pz²)      (3-D range)
 *   ρ   = sqrt(px²+py²)          (horizontal range)
 *
 * Non-zero partial derivatives (Milestone 1 §7.4):
 *
 *   ∂r/∂px  = px/r          ∂r/∂py  = py/r          ∂r/∂pz  = pz/r
 *
 *   ∂θ/∂px  = -py/ρ²        ∂θ/∂py  = px/ρ²         ∂θ/∂pz  = 0
 *
 *   ∂φ/∂px  = -px·pz/(ρ·r²) ∂φ/∂py  = -py·pz/(ρ·r²) ∂φ/∂pz  = ρ/r²
 *
 * Singularity guards match §7.5.1 recommendations.
 * ========================================================================= */
void ekf_compute_jacobian(const double* x, double* Hk) {
    double px = x[0], py = x[4], pz = x[8];

    double r2  = px*px + py*py + pz*pz;
    double r   = sqrt(r2);
    double rho2 = px*px + py*py;
    double rho  = sqrt(rho2);

    /* Regularise to prevent division by zero (Milestone 1 §7.5.1) */
    if (r   < EPSILON_R  ) r   = EPSILON_R;
    if (rho < EPSILON_RHO) rho = EPSILON_RHO;

    /* Recompute safe squared values after clamping */
    double r2_s   = r   * r;
    double rho2_s = rho * rho;

    /* Zero the entire 3×12 Jacobian first */
    memset(Hk, 0, MEAS_DIM * STATE_DIM * sizeof(double));

    /* ── Row 0: ∂r/∂x ─────────────────────────────────────────────────── */
    Hk[0 * STATE_DIM + 0] =  px / r;          /* ∂r/∂px */
    Hk[0 * STATE_DIM + 4] =  py / r;          /* ∂r/∂py */
    Hk[0 * STATE_DIM + 8] =  pz / r;          /* ∂r/∂pz */

    /* ── Row 1: ∂θ/∂x ─────────────────────────────────────────────────── */
    Hk[1 * STATE_DIM + 0] = -py / rho2_s;     /* ∂θ/∂px */
    Hk[1 * STATE_DIM + 4] =  px / rho2_s;     /* ∂θ/∂py */
    /* ∂θ/∂pz = 0 — already zeroed */

    /* ── Row 2: ∂φ/∂x ─────────────────────────────────────────────────── */
    Hk[2 * STATE_DIM + 0] = -(px * pz) / (rho * r2_s);  /* ∂φ/∂px */
    Hk[2 * STATE_DIM + 4] = -(py * pz) / (rho * r2_s);  /* ∂φ/∂py */
    Hk[2 * STATE_DIM + 8] =  rho / r2_s;                 /* ∂φ/∂pz */
}

/* ============================================================================
 * INTERNAL: Measurement noise covariance R (3×3 diagonal)
 *
 * Diagonal entries represent sensor variance in each spherical coordinate:
 *   σ²_r  — range variance       (metres²)
 *   σ²_θ  — azimuth variance     (radians²)
 *   σ²_φ  — elevation variance   (radians²)
 *
 * Values estimated empirically from the dataset by comparing noisy vs true
 * Cartesian positions and propagating through the spherical transform.
 * Ref: Milestone 1 §7.3.7
 * ========================================================================= */
void ekf_init_R(double* R) {
    memset(R, 0, MEAS_DIM * MEAS_DIM * sizeof(double));
    R[0 * MEAS_DIM + 0] = 0.05;    /* σ²_r  — range noise            */
    R[1 * MEAS_DIM + 1] = 0.001;   /* σ²_θ  — azimuth noise          */
    R[2 * MEAS_DIM + 2] = 0.001;   /* σ²_φ  — elevation noise        */
}

/* ============================================================================
 * LIFECYCLE
 * ========================================================================= */

EKF* ekf_create(double dt) {
    EKF* ekf = (EKF*)malloc(sizeof(EKF));
    if (!ekf) {
        utils_log_error("Failed to allocate EKF structure");
        return NULL;
    }

    ekf->dt = dt;

    /* Shared matrices — same constant-jerk model as LKF */
    ekf->F = mat_alloc(STATE_DIM, STATE_DIM);
    ekf->Q = mat_alloc(STATE_DIM, STATE_DIM);
    ekf->R = mat_alloc(MEAS_DIM,  MEAS_DIM);

    if (!ekf->F || !ekf->Q || !ekf->R) {
        utils_log_error("Failed to allocate EKF shared matrices");
        ekf_destroy(ekf);
        return NULL;
    }

    state_init_F(ekf->F, dt);   /* identical to LKF — linear kinematics  */
    state_init_Q(ekf->Q);       /* identical to LKF — same process noise  */
    ekf_init_R  (ekf->R);       /* spherical-coord measurement noise      */

    /* Per-joint state */
    for (int j = 0; j < NUM_JOINTS; j++) {
        ekf->joints[j].x = mat_alloc(STATE_DIM, 1);
        ekf->joints[j].P = mat_alloc(STATE_DIM, STATE_DIM);

        if (!ekf->joints[j].x || !ekf->joints[j].P) {
            utils_log_error("Failed to allocate EKF joint state");
            ekf_destroy(ekf);
            return NULL;
        }

        /* Initial covariance: identity — moderate uncertainty in all states */
        mat_eye(ekf->joints[j].P, STATE_DIM);
    }

    return ekf;
}

void ekf_destroy(EKF* ekf) {
    if (!ekf) return;

    mat_free(ekf->F);
    mat_free(ekf->Q);
    mat_free(ekf->R);

    for (int j = 0; j < NUM_JOINTS; j++) {
        mat_free(ekf->joints[j].x);
        mat_free(ekf->joints[j].P);
    }

    free(ekf);
}

/* ============================================================================
 * CORE: PREDICTION STEP
 *
 * Identical to LKF — state dynamics are linear (constant-jerk model).
 * Ref: Milestone 1 §7.2.1, equations (20) and (21)
 *   x̂_k|k-1 = F · x̂_k-1|k-1
 *   P_k|k-1  = F · P_k-1|k-1 · Fᵀ + Q
 * ========================================================================= */
void ekf_predict(EKF* ekf) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        state_predict_x(ekf->joints[j].x, ekf->F, STATE_DIM);
        state_predict_P(ekf->joints[j].P, ekf->F, ekf->Q, STATE_DIM);
    }
}

/* ============================================================================
 * INTERNAL: EKF update for a single joint
 *
 * Key difference from LKF update:
 *   LKF:  predicted measurement = H · x̂          (linear, constant H)
 *   EKF:  predicted measurement = h(x̂)           (nonlinear function)
 *         innovation covariance uses Hk = ∂h/∂x  (time-varying Jacobian)
 *
 * Steps (Milestone 1 §7.2.2):
 *   1. Convert predicted Cartesian state → spherical: ẑ = h(x̂)
 *   2. Compute Jacobian: Hk = ∂h/∂x|_{x=x̂}
 *   3. Innovation: ν = z_sph - ẑ   (with angle wrapping for θ and φ)
 *   4. Innovation covariance: Sk = Hk·P·Hkᵀ + R
 *   5. Kalman gain: Kk = P·Hkᵀ·Sk⁻¹
 *   6. State update: x̂ = x̂ + Kk·ν
 *   7. Covariance update (Joseph form): P = (I-KkHk)P(I-KkHk)ᵀ + KkRKkᵀ
 * ========================================================================= */

/* Wrap angle to (-π, π] — required for innovation of angular measurements */
static double wrap_angle(double a) {
    /* Uses fast_atan2(sin, cos) identity: always returns value in (-π,π]  */
    double sa = 0.0, ca = 0.0;
    /* Approximate sin/cos via known identities; for wrapping only we use
     * the direct atan2-based wrap which is mathematically exact.           */
    while (a >  PI_VAL) a -= 2.0 * PI_VAL;
    while (a < -PI_VAL) a += 2.0 * PI_VAL;
    return a;
}

static void ekf_update_joint(EKFJointState* joint, const double* z_cart,
                              const double* R) {
    const int n = STATE_DIM;   /* 12 */
    const int m = MEAS_DIM;    /* 3  */

    double* x = joint->x;
    double* P = joint->P;

    /* ── Step 1: Convert Cartesian measurement to spherical ────────────── *
     * z_cart = [px, py, pz] from the CSV (noisy Cartesian position).      *
     * We build a temporary state with just those positions to reuse        *
     * ekf_compute_h — velocity/accel entries are irrelevant to h().       */
    double z_tmp[STATE_DIM];
    memset(z_tmp, 0, STATE_DIM * sizeof(double));
    z_tmp[0] = z_cart[0];   /* px */
    z_tmp[4] = z_cart[1];   /* py */
    z_tmp[8] = z_cart[2];   /* pz */

    double z_sph[3];          /* actual spherical measurement               */
    ekf_compute_h(z_tmp, z_sph);

    /* ── Step 2: Predicted spherical measurement  ẑ = h(x̂) ────────────── */
    double z_pred[3];
    ekf_compute_h(x, z_pred);

    /* ── Step 3: Innovation  ν = z_sph - ẑ  (wrap angular components) ── */
    double nu[3];
    nu[0] = z_sph[0] - z_pred[0];              /* range: no wrapping       */
    nu[1] = wrap_angle(z_sph[1] - z_pred[1]);  /* azimuth: wrap to (-π,π] */
    nu[2] = wrap_angle(z_sph[2] - z_pred[2]);  /* elevation: wrap          */

    /* ── Step 4: Jacobian  Hk = ∂h/∂x|_{x=x̂} ─────────────────────────── */
    double* Hk  = mat_alloc(m, n);   /* 3×12 */
    ekf_compute_jacobian(x, Hk);

    /* ── Step 5: Innovation covariance  Sk = Hk·P·Hkᵀ + R ─────────────── */
    double* Hkt  = mat_alloc(n, m);  /* 12×3 */
    double* PHkt = mat_alloc(n, m);  /* 12×3 */
    double* HkPHkt = mat_alloc(m, m); /* 3×3 */
    double* Sk   = mat_alloc(m, m);  /* 3×3 */

    mat_transpose(Hk, Hkt, m, n);
    mat_mul(P,   Hkt,   PHkt,   n, n, m);
    mat_mul(Hk,  PHkt,  HkPHkt, m, n, m);
    mat_add(HkPHkt, R,  Sk,     m, m);

    /* ── Step 6: Invert Sk (always 3×3 — see Milestone 2 §6.1) ─────────── */
    double Sk_inv[9];
    if (!mat_inverse_3x3(Sk, Sk_inv)) {
        utils_log_error("EKF: singular innovation covariance, skipping update");
        goto cleanup;
    }

    /* ── Step 7: Kalman gain  Kk = P·Hkᵀ·Sk⁻¹ ─────────────────────────── */
    double* Kk = mat_alloc(n, m);   /* 12×3 */
    mat_mul(PHkt, Sk_inv, Kk, n, m, m);

    /* ── Step 8: State update  x̂ = x̂ + Kk·ν ───────────────────────────── */
    double Knu[STATE_DIM];
    mat_mul(Kk, nu, Knu, n, m, 1);
    for (int i = 0; i < n; i++) x[i] += Knu[i];

    /* ── Step 9: Covariance update (Joseph form) ─────────────────────────
     * P = (I - Kk·Hk)·P·(I - Kk·Hk)ᵀ + Kk·R·Kkᵀ
     * Joseph form guarantees positive semi-definiteness (Milestone 1 §7.2.2) */
    mat_joseph_update(P, Kk, Hk, R, n, m);

    mat_free(Kk);

cleanup:
    mat_free(Hk);
    mat_free(Hkt);
    mat_free(PHkt);
    mat_free(HkPHkt);
    mat_free(Sk);
}

/* ============================================================================
 * CORE: UPDATE STEP — dispatches to per-joint update
 *
 * measurements: flat array [NUM_JOINTS × 3] — Cartesian (px,py,pz) per joint
 * Same input layout as LKF for compatibility with lkf_main / ekf_main.
 * ========================================================================= */
void ekf_update(EKF* ekf, const double* measurements) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        const double* z = &measurements[j * MEAS_DIM];
        ekf_update_joint(&ekf->joints[j], z, ekf->R);
    }
}

/* ============================================================================
 * UTILITIES
 * ========================================================================= */

void ekf_set_initial_state(EKF* ekf, int joint_idx, const double* pos) {
    if (joint_idx < 0 || joint_idx >= NUM_JOINTS) return;

    double* x = ekf->joints[joint_idx].x;
    /* Positions known from first measurement; derivatives assumed zero     */
    x[0] = pos[0]; x[1] = 0.0; x[2] = 0.0; x[3] = 0.0;   /* X axis      */
    x[4] = pos[1]; x[5] = 0.0; x[6] = 0.0; x[7] = 0.0;   /* Y axis      */
    x[8] = pos[2]; x[9] = 0.0; x[10]= 0.0; x[11]= 0.0;   /* Z axis      */
}

void ekf_get_positions(const EKF* ekf, double* positions) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        const double* x = ekf->joints[j].x;
        positions[j * 3 + 0] = x[0];   /* px — index 0 */
        positions[j * 3 + 1] = x[4];   /* py — index 4 */
        positions[j * 3 + 2] = x[8];   /* pz — index 8 */
    }
}

void ekf_get_full_state(const EKF* ekf, double* state) {
    for (int j = 0; j < NUM_JOINTS; j++) {
        memcpy(&state[j * STATE_DIM], ekf->joints[j].x,
               STATE_DIM * sizeof(double));
    }
}


In [ ]:
%%writefile ekf_main.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "ekf.h"
#include "io.h"
#include "utils.h"

int main(int argc, char* argv[]) {
    const char* input_file  = (argc > 1) ? argv[1] : "gait_data.csv";
    const char* output_file = (argc > 2) ? argv[2] : "ekf_results.csv";
    double      dt          = (argc > 3) ? atof(argv[3]) : 0.033;

    printf("========================================\n");
    printf("  EXTENDED KALMAN FILTER (EKF)\n");
    printf("========================================\n");
    printf("Input:         %s\n", input_file);
    printf("Output:        %s\n", output_file);
    printf("dt:            %.6f s\n", dt);
    printf("Measurement:   Spherical (r, theta, phi)\n");
    printf("State dim:     %d per joint  (%d total)\n",
           STATE_DIM, TOTAL_STATE_DIM);
    printf("----------------------------------------\n\n");

    /* ── Load noisy Cartesian data ───────────────────────────────────────── */
    MotionData* data = io_load_csv(input_file, NUM_JOINTS);
    if (!data) return 1;

    /* ── Create EKF ──────────────────────────────────────────────────────── */
    EKF* ekf = ekf_create(dt);
    if (!ekf) {
        io_free_motion_data(data);
        return 1;
    }

    /* ── Initialise each joint from the first frame ──────────────────────── */
    for (int j = 0; j < NUM_JOINTS; j++) {
        double pos[3] = {
            data->frames[0][j * 3 + 0],
            data->frames[0][j * 3 + 1],
            data->frames[0][j * 3 + 2]
        };
        ekf_set_initial_state(ekf, j, pos);
    }

    /* ── Open output CSV ─────────────────────────────────────────────────── */
    FILE* out_fp = io_open_output_csv(output_file, NUM_JOINTS, STATE_DIM);
    if (!out_fp) {
        ekf_destroy(ekf);
        io_free_motion_data(data);
        return 1;
    }

    /* ── Allocate output buffer ──────────────────────────────────────────── */
    double* full_state = (double*)malloc(TOTAL_STATE_DIM * sizeof(double));
    if (!full_state) {
        utils_log_error("Failed to allocate output buffer");
        io_close_csv(out_fp);
        ekf_destroy(ekf);
        io_free_motion_data(data);
        return 1;
    }

    /* ── Main processing loop ────────────────────────────────────────────── *
     * Frame 0 : update only (no prior prediction)                          *
     * Frame k : predict → update → write                                   */
    printf("[INFO] Processing %d frames...\n", data->num_frames);

    for (int frame = 0; frame < data->num_frames; frame++) {

        /* Predict (skip on first frame — nothing to predict from yet) */
        if (frame > 0) {
            ekf_predict(ekf);
        }

        /* Update — EKF internally converts Cartesian → spherical,
         * computes Jacobian Hk, then runs the nonlinear update equations  */
        ekf_update(ekf, data->frames[frame]);

        /* Write full 276-D state vector to CSV */
        ekf_get_full_state(ekf, full_state);
        io_write_state_row(out_fp, frame, full_state, TOTAL_STATE_DIM);

        /* Progress indicator (overwrites same line) */
        if ((frame + 1) % 100 == 0 || frame == data->num_frames - 1) {
            printf("\r[INFO] Processed frame %d / %d",
                   frame + 1, data->num_frames);
            fflush(stdout);
        }
    }
    printf("\n\n");

    /* ── Sample results for quick sanity check ───────────────────────────── */
    printf("[INFO] Sample results — Joint 0 (pelvis), final frame:\n");
    double* x = ekf->joints[0].x;
    printf("  Position:     (%.4f, %.4f, %.4f) m\n",   x[0], x[4], x[8]);
    printf("  Velocity:     (%.4f, %.4f, %.4f) m/s\n", x[1], x[5], x[9]);
    printf("  Acceleration: (%.4f, %.4f, %.4f) m/s²\n",x[2], x[6], x[10]);
    printf("  Jerk:         (%.4f, %.4f, %.4f) m/s³\n",x[3], x[7], x[11]);

    /* ── Verify h(x) on the final state for debugging ────────────────────── */
    double h_out[3];
    ekf_compute_h(x, h_out);
    printf("\n  Final predicted measurement h(x̂):\n");
    printf("    Range r     = %.4f m\n",   h_out[0]);
    printf("    Azimuth θ   = %.4f rad\n", h_out[1]);
    printf("    Elevation φ = %.4f rad\n", h_out[2]);

    /* ── Cleanup ─────────────────────────────────────────────────────────── */
    free(full_state);
    io_close_csv(out_fp);
    ekf_destroy(ekf);
    io_free_motion_data(data);

    printf("\n[SUCCESS] EKF results written to %s\n", output_file);
    return 0;
}


In [ ]:
%%bash
gcc -O3 -march=native -mavx2 -mfma -o ekf_main \
    ekf_main.c ekf.c io.c matrix.c state.c \
    atan_utils.c utils.c -lm

./ekf_main "3D Full Body Humain Gait Walking Dataset (Noisy Values).csv" \
    "ekf_results.csv" 0.0333

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  EKF 2D Plots + LKF vs EKF Comparison
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── Settings ────────────────────────────────────────────────────────────
JOINT_IDX = 0
DT        = 1.0 / 30.0
JOINT_NAMES = [
    "pelvis","L5","L3","T12","T8","neck","head",
    "shoulderRight","upperArmRight","forearmRight","handRight",
    "shoulderLeft","upperArmLeft","forearmLeft","handLeft",
    "upperLegRight","lowerLegRight","footRight","toeRight",
    "upperLegLeft","lowerLegLeft","footLeft","toeLeft",
]
JOINT_NAME = JOINT_NAMES[JOINT_IDX]
NUM_JOINTS = 23
STATE_DIM  = 12
NOISY_CSV  = "3D Full Body Humain Gait Walking Dataset (Noisy Values).csv"
TRUE_CSV   = "3D Full Body Humain Gait Walking Dataset (True Values).csv"
LKF_OUT    = "lkf_results.csv"
EKF_OUT    = "ekf_results.csv"

# ── Load all data ────────────────────────────────────────────────────────
def load_raw_csv(path, n_joints=NUM_JOINTS):
    df = pd.read_csv(path, header=None, dtype=str)
    try:
        float(df.iloc[0, 0])
    except (ValueError, TypeError):
        df = df.iloc[1:].reset_index(drop=True)
    cols = n_joints * 3
    data = df.iloc[:, :cols].astype(float).values
    return data.reshape(-1, n_joints, 3)

noisy_arr = load_raw_csv(NOISY_CSV)
true_arr  = load_raw_csv(TRUE_CSV)
lkf_df    = pd.read_csv(LKF_OUT)
ekf_df    = pd.read_csv(EKF_OUT)

N = min(len(lkf_df), len(ekf_df), noisy_arr.shape[0], true_arr.shape[0])
t = np.arange(N) * DT
j = JOINT_IDX

# ── Helper: extract state for a joint from a results dataframe ───────────
def get_axis(df, joint, ax_idx):
    base = 1 + joint * STATE_DIM + ax_idx * 4
    return (df.iloc[:N, base  ].values,   # pos
            df.iloc[:N, base+1].values,   # vel
            df.iloc[:N, base+2].values,   # acc
            df.iloc[:N, base+3].values)   # jerk

lkf_px,lkf_vx,lkf_ax,lkf_jx = get_axis(lkf_df, j, 0)
lkf_py,lkf_vy,lkf_ay,lkf_jy = get_axis(lkf_df, j, 1)
lkf_pz,lkf_vz,lkf_az,lkf_jz = get_axis(lkf_df, j, 2)

ekf_px,ekf_vx,ekf_ax,ekf_jx = get_axis(ekf_df, j, 0)
ekf_py,ekf_vy,ekf_ay,ekf_jy = get_axis(ekf_df, j, 1)
ekf_pz,ekf_vz,ekf_az,ekf_jz = get_axis(ekf_df, j, 2)

print(f"✅ Loaded {N} frames | Joint {JOINT_IDX}: '{JOINT_NAME}'")

# ═══════════════════════════════════════════════════════════════════════════
#  PLOT 1 — EKF State Time-Series (4×3 grid)
# ═══════════════════════════════════════════════════════════════════════════
STATE_DATA_EKF = {
    "Position (m)":        ([ekf_px, ekf_py, ekf_pz], "tab:blue"),
    "Velocity (m/s)":      ([ekf_vx, ekf_vy, ekf_vz], "tab:orange"),
    "Acceleration (m/s²)": ([ekf_ax, ekf_ay, ekf_az], "tab:green"),
    "Jerk (m/s³)":         ([ekf_jx, ekf_jy, ekf_jz], "tab:red"),
}
AXES_LABELS = ["X", "Y", "Z"]

fig, axes = plt.subplots(4, 3, figsize=(16, 14), sharex=True)
fig.suptitle(f"EKF State Estimates — Joint {JOINT_IDX}: {JOINT_NAME}",
             fontsize=15, fontweight="bold", y=1.01)

for row_idx, (ylabel, (series, color)) in enumerate(STATE_DATA_EKF.items()):
    for col_idx, (sig, axis_lbl) in enumerate(zip(series, AXES_LABELS)):
        ax = axes[row_idx][col_idx]
        ax.plot(t, sig, color=color, linewidth=1.2)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(True, alpha=0.3)
        if row_idx == 0:
            ax.set_title(f"{axis_lbl}-axis", fontsize=10, fontweight="bold")
        if row_idx == 3:
            ax.set_xlabel("Time (s)", fontsize=8)

plt.tight_layout()
plt.savefig("ekf_state_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: ekf_state_timeseries.png")

# ═══════════════════════════════════════════════════════════════════════════
#  PLOT 2 — True vs Noisy vs LKF vs EKF Position Comparison
# ═══════════════════════════════════════════════════════════════════════════
true_pos  = true_arr [:N, j, :]
noisy_pos = noisy_arr[:N, j, :]
lkf_pos   = np.column_stack([lkf_px, lkf_py, lkf_pz])
ekf_pos   = np.column_stack([ekf_px, ekf_py, ekf_pz])

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle(f"Position Comparison — Joint {JOINT_IDX}: {JOINT_NAME}",
             fontsize=14, fontweight="bold")

for i, lbl in enumerate(["X", "Y", "Z"]):
    ax = axes[i]
    ax.plot(t, true_pos[:,i],  color="black",      lw=1.5,            label="True",         zorder=4)
    ax.plot(t, noisy_pos[:,i], color="salmon",     lw=0.8, alpha=0.6, label="Noisy",        zorder=1)
    ax.plot(t, lkf_pos[:,i],   color="tab:blue",   lw=1.5, ls="--",   label="LKF estimate", zorder=3)
    ax.plot(t, ekf_pos[:,i],   color="tab:green",  lw=1.5, ls=":",    label="EKF estimate", zorder=2)
    ax.set_ylabel(f"{lbl} position (m)", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)", fontsize=10)
plt.tight_layout()
plt.savefig("ekf_position_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: ekf_position_comparison.png")

# ═══════════════════════════════════════════════════════════════════════════
#  PLOT 3 — LKF vs EKF RMSE across all 23 joints
# ═══════════════════════════════════════════════════════════════════════════
rmse_noisy = np.zeros(NUM_JOINTS)
rmse_lkf   = np.zeros(NUM_JOINTS)
rmse_ekf   = np.zeros(NUM_JOINTS)

for jj in range(NUM_JOINTS):
    b = 1 + jj * STATE_DIM
    lkf_j = np.column_stack([lkf_df.iloc[:N, b].values,
                              lkf_df.iloc[:N, b+4].values,
                              lkf_df.iloc[:N, b+8].values])
    ekf_j = np.column_stack([ekf_df.iloc[:N, b].values,
                              ekf_df.iloc[:N, b+4].values,
                              ekf_df.iloc[:N, b+8].values])
    true_j  = true_arr [:N, jj, :]
    noisy_j = noisy_arr[:N, jj, :]

    rmse_noisy[jj] = np.sqrt(np.mean((noisy_j - true_j)**2))
    rmse_lkf  [jj] = np.sqrt(np.mean((lkf_j   - true_j)**2))
    rmse_ekf  [jj] = np.sqrt(np.mean((ekf_j   - true_j)**2))

x_idx = np.arange(NUM_JOINTS)
width = 0.25

fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(x_idx - width, rmse_noisy, width, color="salmon",     label="Noisy RMSE",
       alpha=0.85, edgecolor="black", linewidth=0.5)
ax.bar(x_idx,         rmse_lkf,   width, color="tab:blue",   label="LKF RMSE",
       alpha=0.85, edgecolor="black", linewidth=0.5)
ax.bar(x_idx + width, rmse_ekf,   width, color="tab:green",  label="EKF RMSE",
       alpha=0.85, edgecolor="black", linewidth=0.5)

ax.set_xticks(x_idx)
ax.set_xticklabels(JOINT_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Position RMSE (m)", fontsize=11)
ax.set_title("LKF vs EKF — Position RMSE across all 23 joints",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

lkf_imp = (rmse_noisy.mean()-rmse_lkf.mean())/rmse_noisy.mean()*100
ekf_imp = (rmse_noisy.mean()-rmse_ekf.mean())/rmse_noisy.mean()*100
ax.text(0.98, 0.96,
        f"LKF improvement: {lkf_imp:.1f}%\nEKF improvement: {ekf_imp:.1f}%",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

plt.tight_layout()
plt.savefig("lkf_ekf_rmse_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📊 RMSE Summary:")
print(f"   Avg Noisy RMSE : {rmse_noisy.mean():.4f} m")
print(f"   Avg LKF   RMSE : {rmse_lkf.mean():.4f} m  ({lkf_imp:.1f}% improvement)")
print(f"   Avg EKF   RMSE : {rmse_ekf.mean():.4f} m  ({ekf_imp:.1f}% improvement)")
print("✅ Saved: lkf_ekf_rmse_comparison.png")

# ═══════════════════════════════════════════════════════════════════════════
#  PLOT 4 — LKF vs EKF State Comparison (vel, acc, jerk)
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 3, figsize=(16, 10), sharex=True)
fig.suptitle(f"LKF vs EKF — Velocity / Acceleration / Jerk\nJoint {JOINT_IDX}: {JOINT_NAME}",
             fontsize=13, fontweight="bold")

rows = [
    ("Velocity (m/s)",      [lkf_vx,lkf_vy,lkf_vz], [ekf_vx,ekf_vy,ekf_vz]),
    ("Acceleration (m/s²)", [lkf_ax,lkf_ay,lkf_az], [ekf_ax,ekf_ay,ekf_az]),
    ("Jerk (m/s³)",         [lkf_jx,lkf_jy,lkf_jz], [ekf_jx,ekf_jy,ekf_jz]),
]

for row_idx, (ylabel, lkf_series, ekf_series) in enumerate(rows):
    for col_idx, axis_lbl in enumerate(AXES_LABELS):
        ax = axes[row_idx][col_idx]
        ax.plot(t, lkf_series[col_idx], color="tab:blue",  lw=1.2, label="LKF")
        ax.plot(t, ekf_series[col_idx], color="tab:green", lw=1.2, ls="--", label="EKF")
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(True, alpha=0.3)
        if row_idx == 0:
            ax.set_title(f"{axis_lbl}-axis", fontsize=10, fontweight="bold")
        if row_idx == 2:
            ax.set_xlabel("Time (s)", fontsize=8)
        if col_idx == 2:
            ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("lkf_ekf_state_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: lkf_ekf_state_comparison.png")
print("\n🎉 All EKF plots complete!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML
import warnings
warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════
#  SETTINGS
# ═══════════════════════════════════════════════════════════════════════════
JOINT_NAMES = [
    "pelvis","L5","L3","T12","T8","neck","head",
    "shoulderRight","upperArmRight","forearmRight","handRight",
    "shoulderLeft","upperArmLeft","forearmLeft","handLeft",
    "upperLegRight","lowerLegRight","footRight","toeRight",
    "upperLegLeft","lowerLegLeft","footLeft","toeLeft",
]
NUM_JOINTS = 23
STATE_DIM  = 12
DT         = 1.0 / 30.0

BONES = [
    (0,1),(1,2),(2,3),(3,4),(4,5),(5,6),
    (4,7),(7,8),(8,9),(9,10),
    (4,11),(11,12),(12,13),(13,14),
    (0,15),(15,16),(16,17),(17,18),
    (0,19),(19,20),(20,21),(21,22),
]

NOISY_CSV = "3D Full Body Humain Gait Walking Dataset (Noisy Values).csv"
LKF_OUT   = "lkf_results.csv"
EKF_OUT   = "ekf_results.csv"

MAX_FRAMES = 3040
FRAME_STEP = 4

# ═══════════════════════════════════════════════════════════════════════════
#  LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════
def load_raw_csv(path, n_joints=NUM_JOINTS):
    df = pd.read_csv(path, header=None, dtype=str)
    try:
        float(df.iloc[0, 0])
    except (ValueError, TypeError):
        df = df.iloc[1:].reset_index(drop=True)
    data = df.iloc[:, :n_joints*3].astype(float).values
    return data.reshape(-1, n_joints, 3)

def load_filter_csv(path, n_joints=NUM_JOINTS, state_dim=STATE_DIM):
    df  = pd.read_csv(path)
    N   = len(df)
    pos = np.zeros((N, n_joints, 3))
    for j in range(n_joints):
        b = 1 + j * state_dim
        pos[:, j, 0] = df.iloc[:, b    ].values
        pos[:, j, 1] = df.iloc[:, b + 4].values
        pos[:, j, 2] = df.iloc[:, b + 8].values
    return pos

print("Loading data...")
noisy_pos = load_raw_csv(NOISY_CSV)
lkf_pos   = load_filter_csv(LKF_OUT)
ekf_pos   = load_filter_csv(EKF_OUT)

N = min(noisy_pos.shape[0], lkf_pos.shape[0], ekf_pos.shape[0], MAX_FRAMES)
frames_idx = list(range(0, N, FRAME_STEP))
n_frames   = len(frames_idx)
print(f"✅ Loaded {N} frames | Animating {n_frames} frames")

# ═══════════════════════════════════════════════════════════════════════════
#  AXIS LIMITS
# ═══════════════════════════════════════════════════════════════════════════
all_pos = np.concatenate([noisy_pos[:N], lkf_pos[:N], ekf_pos[:N]], axis=0)
pad = 0.3
x_min,x_max = all_pos[:,:,0].min()-pad, all_pos[:,:,0].max()+pad
y_min,y_max = all_pos[:,:,1].min()-pad, all_pos[:,:,1].max()+pad
z_min,z_max = all_pos[:,:,2].min()-pad, all_pos[:,:,2].max()+pad

# ═══════════════════════════════════════════════════════════════════════════
#  HELPERS
# ═══════════════════════════════════════════════════════════════════════════
def draw_skeleton(ax, positions, joint_color, bone_color):
    ax.scatter(positions[:,0], positions[:,1], positions[:,2],
               c=joint_color, s=20, depthshade=True, zorder=5)
    for (i, j) in BONES:
        ax.plot([positions[i,0], positions[j,0]],
                [positions[i,1], positions[j,1]],
                [positions[i,2], positions[j,2]],
                color=bone_color, lw=1.8)

def setup_ax(ax, title):
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_zlim(z_min, z_max)
    ax.set_xlabel("X (m)", fontsize=7)
    ax.set_ylabel("Y (m)", fontsize=7)
    ax.set_zlabel("Z (m)", fontsize=7)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.tick_params(labelsize=6)
    ax.view_init(elev=15, azim=-60)

# ═══════════════════════════════════════════════════════════════════════════
#  BUILD ANIMATION
# ═══════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 7))
fig.patch.set_facecolor("#1a1a2e")

ax1 = fig.add_subplot(131, projection='3d')
ax2 = fig.add_subplot(132, projection='3d')
ax3 = fig.add_subplot(133, projection='3d')

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor("#1a1a2e")
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

setup_ax(ax1, "Measured (Noisy)")
setup_ax(ax2, "LKF Estimate")
setup_ax(ax3, "EKF Estimate")

fig.suptitle("3D Full-Body Walking Animation — Measured vs LKF vs EKF",
             fontsize=13, fontweight="bold", color="white", y=1.01)

time_text = fig.text(0.5, 0.97, "", ha="center", fontsize=10, color="white")

def init():
    ax1.cla(); ax2.cla(); ax3.cla()
    setup_ax(ax1, "Measured (Noisy)")
    setup_ax(ax2, "LKF Estimate")
    setup_ax(ax3, "EKF Estimate")
    return []

def update(frame_num):
    fi = frames_idx[frame_num]
    ax1.cla(); ax2.cla(); ax3.cla()
    setup_ax(ax1, "Measured (Noisy)")
    setup_ax(ax2, "LKF Estimate")
    setup_ax(ax3, "EKF Estimate")
    draw_skeleton(ax1, noisy_pos[fi], joint_color="salmon",  bone_color="#ff6b6b")
    draw_skeleton(ax2, lkf_pos  [fi], joint_color="#74b9ff", bone_color="#0984e3")
    draw_skeleton(ax3, ekf_pos  [fi], joint_color="#55efc4", bone_color="#00b894")
    time_text.set_text(f"Time: {fi*DT:.2f}s  |  Frame: {fi}/{N-1}")
    return []

print("Rendering animation — please wait 1-2 minutes...")
anim = animation.FuncAnimation(
    fig, update, frames=n_frames,
    init_func=init, interval=1000/30, blit=False
)

plt.tight_layout()

# ── Save as MP4 ──────────────────────────────────────────────────────────
Writer = animation.FFMpegWriter(fps=30, bitrate=1800,
                                extra_args=['-vcodec','libx264'])
# Add this line before anim.save():
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 50  # MB — for inline display only
anim.save("walking_animation.mp4", writer=Writer, dpi=120,
          savefig_kwargs={'facecolor':'#1a1a2e'})
print("✅ Saved: walking_animation.mp4")

# ── Display inline ────────────────────────────────────────────────────────
plt.close()
HTML(anim.to_jshtml())